In [9]:
from pathlib import Path
import cv2
import pandas as pd
from ultralytics import YOLO
import torch

In [10]:
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Selected device: {DEVICE}")

Selected device: cuda


In [12]:
# -------- CONFIG --------
# VIDEO_PATH = "../data/test.mp4"
# OUT_VIDEO = "../data/tracked_chickens_yolo.mp4"
# OUT_TABLE = "../data/chicken_tracks_yolo.parquet"
OUT_DIR = Path("../sandbox/results/yolox")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# VIDEO_PATH = "../ext-data/test/video_1_5_min.mp4"
VIDEO_PATH = "../data/video/test_1_min.mp4"
OUT_VIDEO = OUT_DIR / "tracked_chickens_yolo.mp4"
OUT_TABLE = OUT_DIR / "chicken_tracks_yolo.parquet"

CONF_THRESH = 0.25  # detection confidence threshold
IOU_THRESH = 0.45  # NMS IoU threshold
TRACKER = "../data/yolo/bytetrack.yaml"  # built-in tracker config
# DEVICE = "mps"                           # set to 0 for GPU if available, else None for CPU
# DEVICE = "cuda"

# Filter detections to these classes (set to None to keep all)
ALLOWED_CLASSES = {"bird"}  # YOLOv8 COCO label; chickens are usually "bird"

# -------- STABLE LABELS MANAGER --------
STABLE_NAMES = ["Chicken A", "Chicken B", "Chicken C"]

# Per-name state
name_state = {
    n: {"track_id": None, "last_pos": None, "last_seen": -1} for n in STABLE_NAMES
}

# Map current tracker IDs -> friendly names
id2name = {}

# Tunable heuristics (adjust to your resolution and motion)
ASSIGN_DIST_THRESH = 400  # pixels; max distance to consider "same chicken"
MISS_TOLERANCE = 60  # frames; how long we keep a slot alive without observations

In [13]:
# -------- UTILITIES --------
def draw_label(img, label, x1, y1):
    """Draw a filled rectangle behind text for readability, positioned above the box."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.6
    thickness = 1
    (w, h), _ = cv2.getTextSize(label, font, scale, thickness)
    top_left = (int(x1), int(y1) - h - 6)
    bottom_right = (int(x1) + w + 6, int(y1))
    cv2.rectangle(img, top_left, bottom_right, (0, 0, 0), -1)
    cv2.putText(
        img,
        label,
        (int(x1) + 3, int(y1) - 4),
        font,
        scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA,
    )


def assign_stable_name(track_id, cx, cy, frame_idx, used_names_this_frame):
    """Return a stable name for this detection and update state.
    If track_id changes, rebind to the nearest existing chicken name.
    """
    # 1) If we already know this track_id, reuse its friendly name.
    if track_id in id2name:
        name = id2name[track_id]
        name_state[name]["last_pos"] = (cx, cy)
        name_state[name]["last_seen"] = frame_idx
        name_state[name]["track_id"] = track_id
        used_names_this_frame.add(name)
        return name

    # 2) Try to match to nearest active name by position (not too old, not already used).
    nearest_name, nearest_dist = None, float("inf")
    for name, st in name_state.items():
        if st["last_pos"] is None:
            continue
        if frame_idx - st["last_seen"] > MISS_TOLERANCE:
            continue
        if name in used_names_this_frame:
            continue
        px, py = st["last_pos"]
        d = ((cx - px) ** 2 + (cy - py) ** 2) ** 0.5
        if d < nearest_dist:
            nearest_name, nearest_dist = name, d

    if nearest_name is not None and nearest_dist <= ASSIGN_DIST_THRESH:
        # Rebind the friendly name from old tracker ID to the new track_id.
        old_id = name_state[nearest_name]["track_id"]
        if old_id is not None and old_id in id2name:
            del id2name[old_id]
        if track_id != -1:  # don't persist -1 in the map
            id2name[track_id] = nearest_name
        name_state[nearest_name]["track_id"] = (
            track_id if track_id != -1 else name_state[nearest_name]["track_id"]
        )
        name_state[nearest_name]["last_pos"] = (cx, cy)
        name_state[nearest_name]["last_seen"] = frame_idx
        used_names_this_frame.add(nearest_name)
        return nearest_name

    # 3) Otherwise, grab an unused/stale slot.
    for name, st in name_state.items():
        if name in used_names_this_frame:
            continue
        if st["last_pos"] is None or frame_idx - st["last_seen"] > MISS_TOLERANCE:
            if track_id != -1:
                id2name[track_id] = name
                name_state[name]["track_id"] = track_id
            name_state[name]["last_pos"] = (cx, cy)
            name_state[name]["last_seen"] = frame_idx
            used_names_this_frame.add(name)
            return name

    # 4) Fallback: raw ID (rare; e.g., >3 detections or unusual frame)
    return f"Chicken {track_id}"

In [14]:
# -------- LOAD MODEL --------
# model = YOLO("yolov8s.pt")
model = YOLO("yolo11x.pt")

In [15]:
# Get video metadata for writer
cap_meta = cv2.VideoCapture(VIDEO_PATH)
fps = cap_meta.get(cv2.CAP_PROP_FPS)
fps = float(fps) if fps and fps > 0 else 30.0
width = int(cap_meta.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap_meta.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_meta.release()

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUT_VIDEO, fourcc, fps, (width, height))

records = []  # will store per-detection per-frame rows
frame_idx = -1


In [16]:
# -------- TRACKING LOOP --------
# The generator yields a Result per frame with tracked boxes (boxes.id).
for result in model.track(
    source=VIDEO_PATH,
    stream=True,
    conf=CONF_THRESH,
    iou=IOU_THRESH,
    tracker=TRACKER,
    persist=True,  # keep tracker state across frames
    device=DEVICE,
):
    used_names_this_frame = set()

    frame_idx += 1
    frame = result.orig_img.copy()  # BGR image

    if result.boxes is None or len(result.boxes) == 0:
        out.write(frame)
        continue

    boxes = result.boxes

    # Iterate detections for this frame

    for b in boxes:
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        w_box = x2 - x1
        h_box = y2 - y1

        conf = float(b.conf[0]) if b.conf is not None else 0.0
        cls_id = int(b.cls[0]) if b.cls is not None else -1
        cls_name = model.names.get(cls_id, "unknown")

        if ALLOWED_CLASSES and cls_name not in ALLOWED_CLASSES:
            continue

        track_id = int(b.id[0]) if hasattr(b, "id") and b.id is not None else -1

        cx = (x1 + x2) / 2.0
        cy = (y1 + y2) / 2.0
        cx_norm = cx / width
        cy_norm = cy / height

        # --- Assign stable friendly name ---
        pretty_name = assign_stable_name(
            track_id, cx, cy, frame_idx, used_names_this_frame
        )

        # Draw
        color = (0, 255, 0)
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        label = f"{pretty_name} conf={conf:.2f} id={track_id}"
        draw_label(frame, label, x1, y1)

        # Save row
        records.append(
            {
                "frame": frame_idx,
                "track_id": track_id,
                "chicken_name": pretty_name,
                "class_id": cls_id,
                "class_name": cls_name,
                "confidence": conf,
                "x1": x1,
                "y1": y1,
                "x2": x2,
                "y2": y2,
                "width": w_box,
                "height": h_box,
                "cx": cx,
                "cy": cy,
                "cx_norm": cx_norm,
                "cy_norm": cy_norm,
            }
        )

    # Write the augmented frame
    out.write(frame)



video 1/1 (frame 1/1495) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/video/test_1_min.mp4: 544x640 (no detections), 12.1ms
video 1/1 (frame 2/1495) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/video/test_1_min.mp4: 544x640 (no detections), 11.9ms
video 1/1 (frame 3/1495) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/video/test_1_min.mp4: 544x640 (no detections), 10.4ms
video 1/1 (frame 4/1495) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/video/test_1_min.mp4: 544x640 (no detections), 10.4ms
video 1/1 (frame 5/1495) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/video/test_1_min.mp4: 544x640 (no detections), 10.4ms
video 1/1 (frame 6/1495) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/video/test_1_min.mp4: 544x640 (no detections), 11.2ms
video 1/1 (frame 7/1495) /home/prince/proj/chicken-behaviour-classifier/notebook/../data/video/test_1_min.mp4: 544x640 (no detections), 11.3m

In [17]:
# -------- SAVE OUTPUTS --------
out.release()

df = pd.DataFrame.from_records(records)
# Helpful indexing for analysis:
df.sort_values(["track_id", "frame"], inplace=True)
df.to_parquet(OUT_TABLE, index=False)  # requires pyarrow

print(f"Saved video to: {OUT_VIDEO}")
print(f"Saved tracks to: {OUT_TABLE}")

Saved video to: ../sandbox/results/yolox/tracked_chickens_yolo.mp4
Saved tracks to: ../sandbox/results/yolox/chicken_tracks_yolo.parquet


In [18]:
df = pd.read_parquet(OUT_TABLE)

In [19]:
df

,frame,track_id,chicken_name,class_id,class_name,confidence,x1,y1,x2,y2,width,height,cx,cy,cx_norm,cy_norm
0,96,-1,Chicken A,14,bird,0.266015,560.259827,411.794434,637.132324,473.982788,76.872498,62.188354,598.696075,442.888611,0.850421,0.768904
1,97,-1,Chicken B,14,bird,0.305667,454.481079,400.389679,632.837769,531.349854,178.356689,130.960175,543.659424,465.869766,0.772243,0.808802
2,101,-1,Chicken A,14,bird,0.285994,535.591187,350.034790,601.458252,459.881104,65.867065,109.846313,568.524719,404.957947,0.807564,0.703052
3,110,-1,Chicken A,14,bird,0.322044,463.495178,301.504059,579.843262,480.749146,116.348083,179.245087,521.669220,391.126602,0.741007,0.679039
4,142,-1,Chicken A,14,bird,0.346941,206.114349,122.436882,347.504761,326.675598,141.390411,204.238716,276.809555,224.556240,0.393195,0.389855
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2426,1490,479,Chicken A,14,bird,0.502697,555.758423,332.852722,657.102722,393.258636,101.344299,60.405914,606.430573,363.055679,0.861407,0.630305
2427,1491,479,Chicken A,14,bird,0.605505,556.013428,332.816162,660.510986,395.334106,104.497559,62.517944,608.262207,364.075134,0.864009,0.632075
2428,1492,479,Chicken A,14,bird,0.600649,553.030334,332.754883,661.408569,397.762299,108.378235,65.007416,607.219452,365.258591,0.862528,0.634129
2429,1493,479,Chicken A,14,bird,0.516436,555.079895,332.994446,659.648132,395.445465,104.568237,62.451019,607.364014,364.219955,0.862733,0.632326


In [20]:
df.chicken_name.unique()

<ArrowStringArray>
[  'Chicken A',   'Chicken B',   'Chicken C',  'Chicken -1',  'Chicken 48',
  'Chicken 63', 'Chicken 169', 'Chicken 431']
Length: 8, dtype: str

In [21]:
df.chicken_name.value_counts()

chicken_name
Chicken B      904
Chicken C      766
Chicken A      745
Chicken 48       9
Chicken -1       2
Chicken 63       2
Chicken 169      2
Chicken 431      1
Name: count, dtype: int64